# CivilityAI: Exploratory Data Analysis & Safety Category Dynamics

This notebook explores the multi-label characteristics of the toxic comment classification corpus, analyzing class imbalances, label co-occurrences, text lengths, and sparsity dynamics.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from safety_ml.data_pipeline import load_raw_dataset, compute_class_imbalance_weights
from safety_ml.settings import CATEGORY_ORDER, RAW_TO_INTERNAL_COLUMN_MAP

### 1. Dataset Ingestion & Schema Normalization
Load raw CSV while preserving original column headers, mapping to internal domain names (`general_toxicity`, `severe_abuse`, `obscene_language`, `threatening_language`, `personal_insult`, `identity_attack`).

In [ ]:
dataset_path = project_root / "datasets" / "source" / "train.csv"
message_frame = load_raw_dataset(dataset_path)
print(f"Total records loaded: {len(message_frame):,}")
message_frame.head()

### 2. Missing Values, Duplicates & Text Lengths

In [ ]:
missing_entries = int(message_frame['comment_text'].isna().sum())
duplicate_messages = int(message_frame.duplicated(subset=['comment_text']).sum())
message_lengths = message_frame['message_body'].str.len()

print(f"Missing message bodies: {missing_entries}")
print(f"Duplicate message count: {duplicate_messages}")
print(f"Average message length: {message_lengths.mean():.1f} chars")
print(f"Maximum message length: {message_lengths.max()} chars")

### 3. Category Frequency & Class Imbalance Analysis

In [ ]:
imbalance_statistics = compute_class_imbalance_weights(message_frame)
category_frequency = imbalance_statistics['category_sample_counts']

for category, count in category_frequency.items():
    pct = imbalance_statistics['prevalence_percentages'][category]
    weight = imbalance_statistics['positive_class_weights'][category]
    print(f"{category:22s}: {count:5d} samples ({pct:5.2f}%) | Recommended pos_weight: {weight:5.2f}")

### 4. Multi-Label Combinations & Co-occurrences

In [ ]:
label_combinations = message_frame[CATEGORY_ORDER].sum(axis=1).value_counts().sort_index()
print("Distribution of concurrent safety categories per message:")
for num_labels, count in label_combinations.items():
    print(f"  {num_labels} active category labels: {count} messages")